# Direct SimPEG Forward Pipeline

This notebook contains the full forward pipeline.

It shows the disk-averaged magnetic scan, electric field, eddy current pattern, conductivity Jacobian, and scar response.

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("XDG_CACHE_HOME", str(Path("figures/.cache").resolve()))
os.environ.setdefault("MPLCONFIGDIR", str(Path("figures/.matplotlib").resolve()))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse, Rectangle

from discretize import TensorMesh
from simpeg import maps
from simpeg.electromagnetics import frequency_domain as fdem

print("Imports loaded.")

## Helper Functions

These cells keep all simulation details inside this notebook.

In [ ]:
def choose_solver():
    """Use Pardiso when available, otherwise use SimPEG-compatible LU."""
    try:
        import scipy.sparse as sp
        from pymatsolver import Pardiso as SelectedSolver

        _ = SelectedSolver(sp.eye(2, format="csr"))
        print("Using solver: Pardiso")
        return SelectedSolver
    except Exception as err:
        print("Pardiso unavailable, using SolverLU. Reason:", repr(err))
        from pymatsolver import SolverLU as SelectedSolver

        return SelectedSolver


def make_equal_area_disk_offsets(coil_diameter_m, n_points=96):
    """
    Uniform equal-weight sample points inside a circular pickup coil disk.
    Give only coil diameter and total number of averaging points.
    """
    if n_points < 1:
        raise ValueError("n_points must be >= 1")

    radius_m = 0.5 * coil_diameter_m

    if n_points == 1:
        return np.array([[0.0, 0.0]], dtype=float)

    i = np.arange(n_points, dtype=float)
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))

    r = radius_m * np.sqrt((i + 0.5) / n_points)
    theta = i * golden_angle

    return np.column_stack((r * np.cos(theta), r * np.sin(theta)))


def make_pickup_disk_receiver_locations(scan_centers_xyz, disk_offsets_xy):
    """
    Expand scan center locations into disk-sampling receiver locations.

    Output ordering:
        center 0 offset 0..n_offsets-1,
        center 1 offset 0..n_offsets-1,
        ...
    """
    scan_centers_xyz = np.asarray(scan_centers_xyz, dtype=float)
    disk_offsets_xy = np.asarray(disk_offsets_xy, dtype=float)

    n_centers = scan_centers_xyz.shape[0]
    n_offsets = disk_offsets_xy.shape[0]

    locations = np.zeros((n_centers * n_offsets, 3), dtype=float)
    locations[:, 0] = np.repeat(scan_centers_xyz[:, 0], n_offsets) + np.tile(disk_offsets_xy[:, 0], n_centers)
    locations[:, 1] = np.repeat(scan_centers_xyz[:, 1], n_offsets) + np.tile(disk_offsets_xy[:, 1], n_centers)
    locations[:, 2] = np.repeat(scan_centers_xyz[:, 2], n_offsets)
    return locations


def elliptic_scar_mask(mesh, tissue_mask, center_xy_m, radius_x_m, radius_y_m):
    """Return a tissue-cell mask for one flat elliptic scar."""
    centers = mesh.cell_centers
    center_xy_m = np.asarray(center_xy_m, dtype=float)
    radius2 = (
        ((centers[:, 0] - center_xy_m[0]) / float(radius_x_m)) ** 2
        + ((centers[:, 1] - center_xy_m[1]) / float(radius_y_m)) ** 2
    )
    return np.asarray(tissue_mask, dtype=bool) & (radius2 <= 1.0)


def make_direct_forward_setup(
    *,
    cell_m=0.001,
    sample_size_3d_m=(0.060, 0.040, 0.001),
    sample_center_3d_m=(0.0, 0.0, -0.005),
    air_x_m=0.012,
    air_y_m=0.012,
    air_below_z_m=0.002,
    air_above_z_m=0.020,
    sigma_air=1e-8,
    sigma_tissue=0.55,
    sigma_scar=0.20,
    frequency_hz=1.0e6,
    excitation_coil_diameter_m=0.025,
    excitation_coil_turns=30,
    excitation_drive_current_a=1.0,
    excitation_coil_segments=128,
    source_normal="z",
    pickup_component="z",
    pickup_coil_diameter_m=0.010,
    pickup_disk_points=16,
    coil_standoff_m=0.0145,
    scan_nx=32,
    scan_ny=20,
    scar_center_xy_m=(0.010, 0.0),
    scar_radius_x_m=0.008,
    scar_radius_y_m=0.004,
    Solver=None,
):
    """Build the default mesh, conductivity models, coil, and scan settings."""
    sample_size_3d_m = np.asarray(sample_size_3d_m, dtype=float)
    sample_center_3d_m = np.asarray(sample_center_3d_m, dtype=float)

    nx_tissue, ny_tissue, nz_tissue = np.round(sample_size_3d_m / cell_m).astype(int)
    nx_air = int(round(air_x_m / cell_m))
    ny_air = int(round(air_y_m / cell_m))
    nz_below = int(round(air_below_z_m / cell_m))
    nz_above = int(round(air_above_z_m / cell_m))

    nx = nx_tissue + 2 * nx_air
    ny = ny_tissue + 2 * ny_air
    nz = nz_below + nz_tissue + nz_above

    hx = np.full(nx, cell_m)
    hy = np.full(ny, cell_m)
    hz = np.full(nz, cell_m)

    sample_size_3d_m = np.array([nx_tissue, ny_tissue, nz_tissue], dtype=float) * cell_m
    sample_bottom_z_m = sample_center_3d_m[2] - 0.5 * sample_size_3d_m[2]
    sample_top_z_m = sample_center_3d_m[2] + 0.5 * sample_size_3d_m[2]
    mesh_origin_m = np.array(
        [
            sample_center_3d_m[0] - 0.5 * sample_size_3d_m[0] - air_x_m,
            sample_center_3d_m[1] - 0.5 * sample_size_3d_m[1] - air_y_m,
            sample_bottom_z_m - air_below_z_m,
        ]
    )

    mesh = TensorMesh([hx, hy, hz], x0=mesh_origin_m)
    centers = mesh.cell_centers
    tissue_mask = (
        (centers[:, 0] >= sample_center_3d_m[0] - 0.5 * sample_size_3d_m[0])
        & (centers[:, 0] <= sample_center_3d_m[0] + 0.5 * sample_size_3d_m[0])
        & (centers[:, 1] >= sample_center_3d_m[1] - 0.5 * sample_size_3d_m[1])
        & (centers[:, 1] <= sample_center_3d_m[1] + 0.5 * sample_size_3d_m[1])
        & (centers[:, 2] >= sample_bottom_z_m)
        & (centers[:, 2] <= sample_top_z_m)
    )

    sigma_healthy = np.full(mesh.n_cells, sigma_air, dtype=float)
    sigma_healthy[tissue_mask] = sigma_tissue

    scar_case = {
        "label": "default scar",
        "center_xy_m": np.asarray(scar_center_xy_m, dtype=float),
        "radius_x_m": float(scar_radius_x_m),
        "radius_y_m": float(scar_radius_y_m),
    }
    scar_mask = elliptic_scar_mask(
        mesh,
        tissue_mask,
        scar_case["center_xy_m"],
        scar_case["radius_x_m"],
        scar_case["radius_y_m"],
    )
    sigma_scar_model = sigma_healthy.copy()
    sigma_scar_model[scar_mask] = sigma_scar

    source_current_a = float(excitation_drive_current_a) * float(excitation_coil_turns)
    source_radius_m = 0.5 * float(excitation_coil_diameter_m)
    pickup_radius_m = 0.5 * float(pickup_coil_diameter_m)
    coil_center_m = np.array(
        [
            sample_center_3d_m[0],
            sample_center_3d_m[1],
            sample_top_z_m + float(coil_standoff_m),
        ]
    )

    if np.any(sigma_healthy <= 0.0):
        raise ValueError("sigma_healthy must be positive because SimPEG uses log(sigma).")

    return {
        "simpeg_mesh": mesh,
        "sigma_healthy": sigma_healthy,
        "sigma_scar_model": sigma_scar_model,
        "model_log_healthy": np.log(sigma_healthy),
        "model_log_scar": np.log(sigma_scar_model),
        "tissue_mask": tissue_mask,
        "scar_mask": scar_mask,
        "scar_case": scar_case,
        "Solver": choose_solver() if Solver is None else Solver,
        "cell_m": float(cell_m),
        "sample_center_3d_m": sample_center_3d_m,
        "sample_size_3d_m": sample_size_3d_m,
        "sample_center_m": sample_center_3d_m[:2].copy(),
        "sample_size_m": sample_size_3d_m[:2].copy(),
        "sample_bottom_z_m": sample_bottom_z_m,
        "sample_top_z_m": sample_top_z_m,
        "air_x_m": float(air_x_m),
        "air_y_m": float(air_y_m),
        "air_below_z_m": float(air_below_z_m),
        "air_above_z_m": float(air_above_z_m),
        "sigma_air": float(sigma_air),
        "sigma_tissue": float(sigma_tissue),
        "sigma_scar": float(sigma_scar),
        "frequency_hz": float(frequency_hz),
        "coil_center_m": coil_center_m,
        "coil_standoff_m": float(coil_standoff_m),
        "source_normal": source_normal,
        "source_radius_m": source_radius_m,
        "source_current_a": source_current_a,
        "excitation_coil_diameter_m": float(excitation_coil_diameter_m),
        "excitation_coil_turns": int(excitation_coil_turns),
        "excitation_coil_segments": int(excitation_coil_segments),
        "excitation_drive_current_a": float(excitation_drive_current_a),
        "pickup_component": pickup_component,
        "pickup_coil_diameter_m": float(pickup_coil_diameter_m),
        "pickup_radius_m": pickup_radius_m,
        "pickup_disk_points": int(pickup_disk_points),
        "pickup_z_m": coil_center_m[2],
        "scan_nx": int(scan_nx),
        "scan_ny": int(scan_ny),
    }

In [ ]:
def make_disk_scan(setup, *, scan_nx=None, scan_ny=None, pickup_disk_points=None):
    """Create scan centers and pickup-disk sample receiver locations."""
    scan_nx = int(setup["scan_nx"] if scan_nx is None else scan_nx)
    scan_ny = int(setup["scan_ny"] if scan_ny is None else scan_ny)
    pickup_disk_points = int(
        setup["pickup_disk_points"] if pickup_disk_points is None else pickup_disk_points
    )

    sample_center_m = np.asarray(setup["sample_center_m"], dtype=float)
    sample_size_m = np.asarray(setup["sample_size_m"], dtype=float)
    scan_x = np.linspace(
        sample_center_m[0] - 0.5 * sample_size_m[0],
        sample_center_m[0] + 0.5 * sample_size_m[0],
        scan_nx,
    )
    scan_y = np.linspace(
        sample_center_m[1] - 0.5 * sample_size_m[1],
        sample_center_m[1] + 0.5 * sample_size_m[1],
        scan_ny,
    )
    scan_X_m, scan_Y_m = np.meshgrid(scan_x, scan_y, indexing="xy")
    scan_centers_m = np.column_stack(
        [
            scan_X_m.ravel(),
            scan_Y_m.ravel(),
            np.full(scan_X_m.size, setup["pickup_z_m"]),
        ]
    )
    disk_offsets_xy_m = make_equal_area_disk_offsets(
        setup["pickup_coil_diameter_m"],
        n_points=pickup_disk_points,
    )
    receiver_locations_m = make_pickup_disk_receiver_locations(
        scan_centers_m,
        disk_offsets_xy_m,
    )

    return {
        "scan_x": scan_x,
        "scan_y": scan_y,
        "scan_X_m": scan_X_m,
        "scan_Y_m": scan_Y_m,
        "scan_centers_m": scan_centers_m,
        "disk_offsets_xy_m": disk_offsets_xy_m,
        "receiver_locations_m": receiver_locations_m,
        "n_centers": scan_centers_m.shape[0],
        "n_disk": pickup_disk_points,
        "shape": (scan_ny, scan_nx),
    }


def make_disk_simulation(setup, scan):
    """Build one SimPEG disk-averaged magnetic-flux-density simulation."""
    mesh = setup["simpeg_mesh"]
    locations = np.asarray(scan["receiver_locations_m"], dtype=float)
    if not np.all(mesh.is_inside(locations)):
        raise ValueError("Some receiver locations are outside the mesh.")

    receivers = [
        fdem.receivers.PointMagneticFluxDensitySecondary(
            locations,
            orientation=setup["pickup_component"],
            component="real",
        ),
        fdem.receivers.PointMagneticFluxDensitySecondary(
            locations,
            orientation=setup["pickup_component"],
            component="imag",
        ),
    ]
    source = fdem.sources.CircularLoop(
        receiver_list=receivers,
        frequency=setup["frequency_hz"],
        location=setup["coil_center_m"],
        orientation=setup["source_normal"],
        radius=setup["source_radius_m"],
        current=setup["source_current_a"],
    )
    survey = fdem.Survey([source])
    simulation = fdem.simulation.Simulation3DMagneticFluxDensity(
        mesh=mesh,
        survey=survey,
        sigmaMap=maps.ExpMap(mesh),
        solver=setup["Solver"],
    )
    return simulation


def disk_average_raw_data(raw, n_centers, n_disk):
    """Average raw SimPEG real/imag disk samples back to one scan value."""
    raw = np.asarray(raw)
    n_centers = int(n_centers)
    n_disk = int(n_disk)
    n_locations = n_centers * n_disk
    expected = 2 * n_locations
    if raw.size != expected:
        raise ValueError(f"Expected {expected} receiver values, got {raw.size}.")

    real_T = raw[:n_locations].reshape(n_centers, n_disk).mean(axis=1)
    imag_T = raw[n_locations:].reshape(n_centers, n_disk).mean(axis=1)
    return real_T, imag_T


def run_forward(simulation, model_log_sigma, scan):
    """Run one forward model and return fields plus disk-averaged data."""
    fields = simulation.fields(model_log_sigma)
    raw = simulation.dpred(model_log_sigma, f=fields)
    real_T, imag_T = disk_average_raw_data(raw, scan["n_centers"], scan["n_disk"])
    shape = scan["shape"]
    return {
        "fields": fields,
        "raw": raw,
        "real_T": real_T,
        "imag_T": imag_T,
        "real_nT": real_T * 1e9,
        "imag_nT": imag_T * 1e9,
        "real_grid_nT": (real_T * 1e9).reshape(shape),
        "imag_grid_nT": (imag_T * 1e9).reshape(shape),
    }

In [ ]:
def cell_center_electric_and_current(simulation, fields, sigma_full):
    """Move SimPEG edge E to cell centers and compute J = sigma E."""
    source = simulation.survey.source_list[0]
    edge_e = np.asarray(fields[source, "e"]).squeeze()
    E_cell = (simulation.mesh.aveE2CCV @ edge_e).reshape(
        (simulation.mesh.n_cells, 3),
        order="F",
    )
    sigma_full = np.asarray(sigma_full)
    J_cell = sigma_full[:, np.newaxis] * E_cell
    return {
        "E_cell": E_cell,
        "J_cell": J_cell,
        "E_abs": np.linalg.norm(E_cell, axis=1),
        "J_abs": np.linalg.norm(J_cell, axis=1),
        "E_imag_abs": np.linalg.norm(np.imag(E_cell), axis=1),
        "J_imag_abs": np.linalg.norm(np.imag(J_cell), axis=1),
    }


def cell_values_to_xy_grid(mesh, cell_mask, values):
    """Average cell values onto an x-y grid, usually through the tissue layer."""
    cell_mask = np.asarray(cell_mask, dtype=bool)
    values = np.asarray(values)
    if values.shape[0] == mesh.n_cells:
        local_values = values[cell_mask]
    elif values.shape[0] == int(cell_mask.sum()):
        local_values = values
    else:
        raise ValueError("values must have length mesh.n_cells or cell_mask.sum().")

    centers = mesh.cell_centers[cell_mask]
    x_key = np.round(centers[:, 0], 12)
    y_key = np.round(centers[:, 1], 12)
    x_unique = np.unique(x_key)
    y_unique = np.unique(y_key)
    x_index = {value: idx for idx, value in enumerate(x_unique)}
    y_index = {value: idx for idx, value in enumerate(y_unique)}

    dtype = np.result_type(local_values, float)
    sums = np.zeros((y_unique.size, x_unique.size), dtype=dtype)
    counts = np.zeros((y_unique.size, x_unique.size), dtype=float)
    for x_value, y_value, value in zip(x_key, y_key, local_values):
        iy = y_index[y_value]
        ix = x_index[x_value]
        sums[iy, ix] += value
        counts[iy, ix] += 1.0

    grid = np.full_like(sums, np.nan, dtype=dtype)
    filled = counts > 0.0
    grid[filled] = sums[filled] / counts[filled]
    dx = float(np.median(np.diff(x_unique))) if x_unique.size > 1 else float(mesh.h[0][0])
    dy = float(np.median(np.diff(y_unique))) if y_unique.size > 1 else float(mesh.h[1][0])
    extent_cm = [
        (x_unique.min() - 0.5 * dx) * 100.0,
        (x_unique.max() + 0.5 * dx) * 100.0,
        (y_unique.min() - 0.5 * dy) * 100.0,
        (y_unique.max() + 0.5 * dy) * 100.0,
    ]
    return {
        "grid": grid,
        "extent_cm": extent_cm,
        "x_m": x_unique,
        "y_m": y_unique,
        "x_cm": x_unique * 100.0,
        "y_cm": y_unique * 100.0,
    }


def vector_xy_grids(mesh, cell_mask, vector_values, *, part="imag"):
    """Return x and y component grids for a vector field."""
    vector_values = np.asarray(vector_values)
    if part == "imag":
        vectors = np.imag(vector_values)
    elif part == "real":
        vectors = np.real(vector_values)
    elif part == "abs":
        vectors = np.abs(vector_values)
    else:
        raise ValueError("part must be 'imag', 'real', or 'abs'.")

    x_info = cell_values_to_xy_grid(mesh, cell_mask, vectors[:, 0])
    y_info = cell_values_to_xy_grid(mesh, cell_mask, vectors[:, 1])
    mag_info = cell_values_to_xy_grid(
        mesh,
        cell_mask,
        np.linalg.norm(vectors[:, :2], axis=1),
    )
    return {
        "x_grid": x_info["grid"],
        "y_grid": y_info["grid"],
        "mag_grid": mag_info["grid"],
        "extent_cm": mag_info["extent_cm"],
        "x_cm": mag_info["x_cm"],
        "y_cm": mag_info["y_cm"],
    }

In [ ]:
def make_source_pixels_xy(mesh, source_mask, pixel_size_m):
    """Group source cells into square x-y pixels for Jacobian columns."""
    source_mask = np.asarray(source_mask, dtype=bool)
    source_indices = np.flatnonzero(source_mask)
    centers = mesh.cell_centers[source_indices]
    x = centers[:, 0]
    y = centers[:, 1]

    dx = float(np.min(mesh.h[0]))
    dy = float(np.min(mesh.h[1]))
    x_edges = np.arange(x.min() - 0.5 * dx, x.max() + 0.5 * dx + pixel_size_m, pixel_size_m)
    y_edges = np.arange(y.min() - 0.5 * dy, y.max() + 0.5 * dy + pixel_size_m, pixel_size_m)

    pixel_indices = []
    pixel_centers_xy = []
    pixel_n = []
    for ix in range(len(x_edges) - 1):
        for iy in range(len(y_edges) - 1):
            local = (
                (x >= x_edges[ix])
                & (x < x_edges[ix + 1])
                & (y >= y_edges[iy])
                & (y < y_edges[iy + 1])
            )
            if not np.any(local):
                continue
            full_indices = source_indices[local]
            pixel_indices.append(full_indices)
            pixel_centers_xy.append(
                [
                    0.5 * (x_edges[ix] + x_edges[ix + 1]),
                    0.5 * (y_edges[iy] + y_edges[iy + 1]),
                ]
            )
            pixel_n.append(full_indices.size)

    return {
        "pixel_indices": pixel_indices,
        "pixel_xy": np.asarray(pixel_centers_xy, dtype=float),
        "pixel_n": np.asarray(pixel_n, dtype=int),
        "x_edges_m": x_edges,
        "y_edges_m": y_edges,
    }


def build_jacobian_sigma(
    simulation,
    model_log_sigma,
    fields,
    sigma_reference,
    pixels,
    scan,
    *,
    column_indices=None,
    progress_every=10,
):
    """Build J = d Im(Bz_disk) / d sigma for selected source pixels."""
    all_pixel_indices = pixels["pixel_indices"]
    if column_indices is None:
        column_indices = np.arange(len(all_pixel_indices))
    else:
        column_indices = np.asarray(column_indices, dtype=int)

    n_measurements = int(scan["n_centers"])
    J_sigma = np.zeros((n_measurements, column_indices.size), dtype=float)
    sigma_reference = np.asarray(sigma_reference, dtype=float)

    for out_col, pixel_col in enumerate(column_indices):
        if progress_every and (
            (out_col + 1) % progress_every == 0 or (out_col + 1) == column_indices.size
        ):
            print(f"Jacobian column {out_col + 1}/{column_indices.size}")

        cell_indices = all_pixel_indices[int(pixel_col)]
        dlog_sigma = np.zeros(simulation.mesh.n_cells, dtype=float)
        dlog_sigma[cell_indices] = 1.0 / sigma_reference[cell_indices]
        raw_j = simulation.Jvec(model_log_sigma, dlog_sigma, f=fields)
        _, imag_T = disk_average_raw_data(raw_j, scan["n_centers"], scan["n_disk"])
        J_sigma[:, out_col] = imag_T * 1e9

    return {
        "J_sigma": J_sigma,
        "pixel_column_indices": column_indices,
        "pixel_xy": pixels["pixel_xy"][column_indices],
        "pixel_indices": [all_pixel_indices[int(idx)] for idx in column_indices],
        "pixel_n": pixels["pixel_n"][column_indices],
        "unit": "nT per S/m",
    }


def pixel_delta_sigma(jacobian, sigma_base, sigma_new):
    """Average a conductivity change over each Jacobian source pixel."""
    sigma_base = np.asarray(sigma_base, dtype=float)
    sigma_new = np.asarray(sigma_new, dtype=float)
    return np.array(
        [
            float(np.mean(sigma_new[cell_indices] - sigma_base[cell_indices]))
            for cell_indices in jacobian["pixel_indices"]
        ]
    )


def predict_linearized_change(jacobian, sigma_base, sigma_new):
    """Use the Jacobian to predict the data change from a new sigma model."""
    delta_sigma = pixel_delta_sigma(jacobian, sigma_base, sigma_new)
    return jacobian["J_sigma"] @ delta_sigma, delta_sigma

In [ ]:
def _sample_rect_cm(setup):
    center_cm = np.asarray(setup["sample_center_m"]) * 100.0
    size_cm = np.asarray(setup["sample_size_m"]) * 100.0
    return (
        center_cm[0] - 0.5 * size_cm[0],
        center_cm[1] - 0.5 * size_cm[1],
        size_cm[0],
        size_cm[1],
    )


def add_tissue_outline(ax, setup, *, edgecolor="black", linewidth=1.2):
    """Draw the tissue footprint in x-y."""
    x0, y0, width, height = _sample_rect_cm(setup)
    ax.add_patch(
        Rectangle(
            (x0, y0),
            width,
            height,
            fill=False,
            edgecolor=edgecolor,
            linewidth=linewidth,
            linestyle="--",
        )
    )


def add_scar_outline(ax, setup, *, edgecolor="black", linewidth=1.2):
    """Draw the default scar footprint in x-y."""
    scar_case = setup["scar_case"]
    center_xy_cm = np.asarray(scar_case["center_xy_m"]) * 100.0
    ax.add_patch(
        Ellipse(
            center_xy_cm,
            2.0 * scar_case["radius_x_m"] * 100.0,
            2.0 * scar_case["radius_y_m"] * 100.0,
            fill=False,
            edgecolor=edgecolor,
            linewidth=linewidth,
        )
    )


def add_source_outline(ax, setup, *, edgecolor="tab:red", linewidth=1.3):
    """Draw the source coil projection in x-y."""
    center_cm = np.asarray(setup["coil_center_m"][:2]) * 100.0
    ax.add_patch(
        Circle(
            center_cm,
            setup["source_radius_m"] * 100.0,
            fill=False,
            edgecolor=edgecolor,
            linewidth=linewidth,
        )
    )


def _symmetric_limits(values):
    vmax = float(np.nanpercentile(np.abs(values), 99))
    vmax = max(vmax, 1e-30)
    return -vmax, vmax


def plot_scan_grid(
    scan,
    grid_nT,
    setup,
    *,
    title,
    colorbar_label="Im(Bz) [nT]",
    symmetric=True,
    show_scar=False,
    cmap="RdBu_r",
    figsize=(6.8, 4.8),
):
    """Plot one scan map."""
    grid_nT = np.asarray(grid_nT)
    fig, ax = plt.subplots(figsize=figsize)
    if symmetric:
        vmin, vmax = _symmetric_limits(grid_nT)
    else:
        vmin = vmax = None
    image = ax.pcolormesh(
        scan["scan_X_m"] * 100.0,
        scan["scan_Y_m"] * 100.0,
        grid_nT,
        shading="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    add_tissue_outline(ax, setup)
    add_source_outline(ax, setup)
    if show_scar:
        add_scar_outline(ax, setup)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title)
    ax.set_xlabel("x [cm]")
    ax.set_ylabel("y [cm]")
    fig.colorbar(image, ax=ax, label=colorbar_label, shrink=0.84)
    plt.tight_layout()
    plt.show()
    return fig, ax


def plot_cell_grid(
    grid_info,
    setup,
    *,
    title,
    colorbar_label,
    symmetric=False,
    quiver=None,
    quiver_step=4,
    cmap="viridis",
    figsize=(6.8, 4.8),
):
    """Plot a tissue x-y cell grid, optionally with x-y vectors."""
    grid = np.asarray(grid_info["grid"])
    fig, ax = plt.subplots(figsize=figsize)
    if symmetric:
        vmin, vmax = _symmetric_limits(grid)
    else:
        vmin = vmax = None
    image = ax.imshow(
        grid,
        origin="lower",
        extent=grid_info["extent_cm"],
        aspect="equal",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    if quiver is not None:
        x_grid, y_grid = np.meshgrid(grid_info["x_cm"], grid_info["y_cm"])
        u = np.nan_to_num(quiver[0])
        v = np.nan_to_num(quiver[1])
        step = max(1, int(quiver_step))
        ax.quiver(
            x_grid[::step, ::step],
            y_grid[::step, ::step],
            u[::step, ::step],
            v[::step, ::step],
            color="black",
            pivot="middle",
            linewidth=0.35,
            scale=None,
        )
    add_tissue_outline(ax, setup)
    add_source_outline(ax, setup)
    ax.set_title(title)
    ax.set_xlabel("x [cm]")
    ax.set_ylabel("y [cm]")
    fig.colorbar(image, ax=ax, label=colorbar_label, shrink=0.84)
    plt.tight_layout()
    plt.show()
    return fig, ax


def pixel_values_to_grid(pixel_xy, values):
    """Put source-pixel values on a regular x-y grid for plotting."""
    pixel_xy = np.asarray(pixel_xy, dtype=float)
    values = np.asarray(values)
    x_key = np.round(pixel_xy[:, 0], 12)
    y_key = np.round(pixel_xy[:, 1], 12)
    x_unique = np.unique(x_key)
    y_unique = np.unique(y_key)
    grid = np.full((y_unique.size, x_unique.size), np.nan, dtype=np.result_type(values, float))
    for value, x_value, y_value in zip(values, x_key, y_key):
        ix = int(np.where(x_unique == x_value)[0][0])
        iy = int(np.where(y_unique == y_value)[0][0])
        grid[iy, ix] = value

    dx = float(np.median(np.diff(x_unique))) if x_unique.size > 1 else 0.001
    dy = float(np.median(np.diff(y_unique))) if y_unique.size > 1 else 0.001
    extent_cm = [
        (x_unique.min() - 0.5 * dx) * 100.0,
        (x_unique.max() + 0.5 * dx) * 100.0,
        (y_unique.min() - 0.5 * dy) * 100.0,
        (y_unique.max() + 0.5 * dy) * 100.0,
    ]
    return {"grid": grid, "extent_cm": extent_cm, "x_cm": x_unique * 100.0, "y_cm": y_unique * 100.0}


def plot_jacobian_column(scan, jacobian, setup, *, column=None):
    """Show one Jacobian source pixel and its scan sensitivity."""
    J_sigma = jacobian["J_sigma"]
    if column is None:
        column = int(np.argmax(np.linalg.norm(J_sigma, axis=0)))
    column = int(column)
    scan_grid = J_sigma[:, column].reshape(scan["shape"])

    source_values = np.zeros(jacobian["pixel_xy"].shape[0], dtype=float)
    source_values[column] = 1.0
    source_info = pixel_values_to_grid(jacobian["pixel_xy"], source_values)

    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))

    vmin, vmax = _symmetric_limits(scan_grid)
    image = axes[0].pcolormesh(
        scan["scan_X_m"] * 100.0,
        scan["scan_Y_m"] * 100.0,
        scan_grid,
        shading="auto",
        cmap="RdBu_r",
        vmin=vmin,
        vmax=vmax,
    )
    add_tissue_outline(axes[0], setup)
    add_source_outline(axes[0], setup)
    axes[0].set_aspect("equal", adjustable="box")
    axes[0].set_title("Jacobian data column")
    axes[0].set_xlabel("x [cm]")
    axes[0].set_ylabel("y [cm]")
    fig.colorbar(image, ax=axes[0], label=jacobian["unit"], shrink=0.82)

    axes[1].imshow(
        source_info["grid"],
        origin="lower",
        extent=source_info["extent_cm"],
        aspect="equal",
        cmap="Greys",
        vmin=0.0,
        vmax=1.0,
    )
    add_tissue_outline(axes[1], setup)
    axes[1].set_title("Source pixel")
    axes[1].set_xlabel("x [cm]")
    axes[1].set_ylabel("y [cm]")

    plt.tight_layout()
    plt.show()
    return fig, axes

## 1. Setup

This builds the mesh, tissue, coil, pickup disk, and one scar model.

Change these numbers when you want a new case.

In [ ]:
CELL_M = 0.001  # use 0.002 for a faster test run

setup = make_direct_forward_setup(
    cell_m=CELL_M,
    scan_nx=32,
    scan_ny=20,
    pickup_disk_points=16,
)

scan = make_disk_scan(setup)
simulation = make_disk_simulation(setup, scan)
mesh = setup["simpeg_mesh"]

print("mesh shape:", mesh.shape_cells)
print("mesh cells:", mesh.n_cells)
print("tissue cells:", int(setup["tissue_mask"].sum()))
print("scar cells:", int(setup["scar_mask"].sum()))
print("scan shape:", scan["shape"])
print("pickup disk points:", scan["n_disk"])
print("coil center [cm]:", setup["coil_center_m"] * 100.0)
print("source current [A]:", setup["source_current_a"])

## 2. Healthy Forward Run

This is the baseline disk-averaged magnetic response.

In [ ]:
healthy = run_forward(simulation, setup["model_log_healthy"], scan)

plot_scan_grid(
    scan,
    healthy["imag_grid_nT"],
    setup,
    title="Healthy disk-averaged Im(Bz)",
    colorbar_label="Im(Bz) [nT]",
)

print("healthy Im(Bz) min/max [nT]:", float(healthy["imag_nT"].min()), float(healthy["imag_nT"].max()))

## 3. Electric Field and Eddy Current

SimPEG gives the electric field on mesh edges.

The package moves it to cell centers and computes `J = sigma E`.

In [ ]:
fields_healthy = cell_center_electric_and_current(
    simulation,
    healthy["fields"],
    setup["sigma_healthy"],
)

E_grid = cell_values_to_xy_grid(
    mesh,
    setup["tissue_mask"],
    fields_healthy["E_abs"],
)
J_imag = vector_xy_grids(
    mesh,
    setup["tissue_mask"],
    fields_healthy["J_cell"],
    part="imag",
)
J_grid = {
    "grid": J_imag["mag_grid"],
    "extent_cm": J_imag["extent_cm"],
    "x_cm": J_imag["x_cm"],
    "y_cm": J_imag["y_cm"],
}

plot_cell_grid(
    E_grid,
    setup,
    title="Electric field in tissue",
    colorbar_label="|E| [V/m]",
)
plot_cell_grid(
    J_grid,
    setup,
    title="Eddy current pattern in tissue",
    colorbar_label="|Im(Jxy)| [A/m2]",
    quiver=(J_imag["x_grid"], J_imag["y_grid"]),
    quiver_step=5,
)

print("max |E| in tissue [V/m]:", float(np.nanmax(E_grid["grid"])))
print("max |Im(Jxy)| in tissue [A/m2]:", float(np.nanmax(J_grid["grid"])))

## 4. Jacobian

The Jacobian tells how the scan changes when one conductivity pixel changes.

The default pixels are coarse so this cell runs faster. Use `0.001` for 1 mm pixels.

In [ ]:
JACOBIAN_PIXEL_SIZE_M = 0.010

pixels = make_source_pixels_xy(
    mesh,
    setup["tissue_mask"],
    JACOBIAN_PIXEL_SIZE_M,
)

jacobian = build_jacobian_sigma(
    simulation,
    setup["model_log_healthy"],
    healthy["fields"],
    setup["sigma_healthy"],
    pixels,
    scan,
    progress_every=5,
)

linear_change_nT, delta_sigma_pixel = predict_linearized_change(
    jacobian,
    setup["sigma_healthy"],
    setup["sigma_scar_model"],
)
linear_change_grid_nT = linear_change_nT.reshape(scan["shape"])

plot_jacobian_column(scan, jacobian, setup)
plot_scan_grid(
    scan,
    linear_change_grid_nT,
    setup,
    title="Jacobian prediction: scar - healthy",
    colorbar_label="Delta Im(Bz) [nT]",
    show_scar=True,
)

print("Jacobian shape [measurements x pixels]:", jacobian["J_sigma"].shape)
print("Jacobian unit:", jacobian["unit"])
print("source pixel size [mm]:", JACOBIAN_PIXEL_SIZE_M * 1000.0)
print("nonzero scar pixels:", int(np.count_nonzero(np.abs(delta_sigma_pixel) > 0.0)))
print("Jacobian prediction min/max [nT]:", float(linear_change_grid_nT.min()), float(linear_change_grid_nT.max()))

## 5. Scar Forward Run

This direct run shows the full scar response.

Compare it with the Jacobian prediction above.

In [ ]:
scar = run_forward(simulation, setup["model_log_scar"], scan)
scar_change_grid_nT = scar["imag_grid_nT"] - healthy["imag_grid_nT"]

plot_scan_grid(
    scan,
    scar_change_grid_nT,
    setup,
    title="Direct scar change: scar - healthy",
    colorbar_label="Delta Im(Bz) [nT]",
    show_scar=True,
)

print("direct scar change min/max [nT]:", float(scar_change_grid_nT.min()), float(scar_change_grid_nT.max()))

## Reuse

To reuse the pipeline:

1. Change the values in `make_direct_forward_setup(...)`.
2. Put your conductivity model in a full mesh array called `sigma_new`.
3. Run `run_forward(simulation, np.log(sigma_new), scan)`.
4. Build pixels with `make_source_pixels_xy(...)`.
5. Build `jacobian` with `build_jacobian_sigma(...)`.
6. Use `predict_linearized_change(...)` for a fast first-order response.